## Generating Ground Truth Data

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../01_module_agentic_rag/.env")
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

In [2]:
model = "qwen/qwen3.5-9b"

In [3]:
from ingest import load_faq_data
documents = load_faq_data()

In [4]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [5]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

118

In [6]:
# assign llm course to documents var
documents = documents_llm

In [7]:
len(documents)

118

In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.

Use casual language
""".strip()

In [10]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [11]:
import json

user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.chat.completions.parse(
    model=model,
    messages=messages,
    response_format=Questions
)

In [15]:
response

ParsedChatCompletion[TypeVar](id='chatcmpl-dtiqnwa34owmwji1px3b8', choices=[ParsedChoice[TypeVar](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[TypeVar](content='{\n  "questions": [\n    "I found the LLM Zoomcamp course late. Is it too late to join now?",\n    "Can I still sign up if I discovered this course just recently?",\n    "Is there a deadline for joining, or can I enroll even though I\'m late?",\n    "If I join now, will I miss out on anything important like the project submission?",\n    "Does signing up late mean I won\'t be able to get my certificate later?"\n  ]\n}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, parsed=Questions(questions=['I found the LLM Zoomcamp course late. Is it too late to join now?', 'Can I still sign up if I discovered this course just recently?', "Is there a deadline for joining, or can I enroll even though I'm late?", 'If I join now, will I miss out on anythin

In [16]:
results = response.choices[0].message.parsed
results

Questions(questions=['I found the LLM Zoomcamp course late. Is it too late to join now?', 'Can I still sign up if I discovered this course just recently?', "Is there a deadline for joining, or can I enroll even though I'm late?", 'If I join now, will I miss out on anything important like the project submission?', "Does signing up late mean I won't be able to get my certificate later?"])

In [17]:
results.questions

['I found the LLM Zoomcamp course late. Is it too late to join now?',
 'Can I still sign up if I discovered this course just recently?',
 "Is there a deadline for joining, or can I enroll even though I'm late?",
 'If I join now, will I miss out on anything important like the project submission?',
 "Does signing up late mean I won't be able to get my certificate later?"]

In [18]:
usage_dict = {
    "prompt_tokens": response.usage.prompt_tokens,
    "total_tokens": response.usage.total_tokens
}

In [19]:
usage_dict['prompt_tokens'], usage_dict['total_tokens']

(191, 305)

In [20]:
from evaluation_utils import llm_structured_chat_completions, calc_price_chat_completions

In [21]:
result, usage = llm_structured_chat_completions(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

In [22]:
cost = calc_price_chat_completions(usage)

cost

TypeError: 'CompletionUsage' object is not subscriptable

In [23]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course LLM Zoomcamp, is it too late for me to join now?',
  'document': '74eb249bbf'},
 {'question': 'If I sign up today, can I still get the certificate if the project submission deadline is coming soon?',
  'document': '74eb249bbf'},
 {'question': 'Is there really a deadline to submit the project if I want that official certificate at the end?',
  'document': '74eb249bbf'},
 {'question': "I'm interested in the course but worried about missing out, does joining late mean I miss the cert?",
  'document': '74eb249bbf'},
 {'question': 'Can anyone who joins this zoomcamp get a certificate as long as they finish their project in time?',
  'document': '74eb249bbf'}]

In [24]:
from evaluation_utils import llm_structured_retry_chat_completions

In [25]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry_chat_completions(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [26]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [27]:
# parallel processing
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [28]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/118 [00:00<?, ?it/s]

In [29]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

583

In [ ]:
from evaluation_utils import calc_price_chat_completions

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

In [30]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth.shape

(583, 2)

In [32]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)